### This is a simple chatbot agent (single chain), where the replies from LLM is restricted to its knowledge base
##### Suppose we need to know some real time data, like stock price. This agent wont reply. So in that case we need multi agent approach, where it connect to tools for answering

In [ ]:
from typing import Annotated
from dotenv import load_dotenv
load_dotenv() # load environment variables from a .env file which contains the API key. Eg: Gemini, OpenAI etc

from typing_extensions import TypedDict
from langchain.chat_models import init_chat_model
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages

In [2]:
from langchain.chat_models import init_chat_model

In [ ]:
# testing the llm access

llm = init_chat_model("google_genai:gemini-2.0-flash")
llm.invoke("Who was the first president of the USA?")

In [ ]:
llm = init_chat_model("google_genai:gemini-2.0-flash")

class State(TypedDict):
    messages: Annotated[list, add_messages] # add_messages is a special annotation to indicate that this field is a list of messages for the chatbot. 
                                            # It is a special reducer function to preserve the history

def chatbot(state: State) -> State:
    return {"messages": [llm.invoke(state["messages"])]}  #chatbot is a node which accepts messages and returns the response from llm

builder = StateGraph(State)
builder.add_node("chatbot_node", chatbot)

builder.add_edge(START, "chatbot_node")
builder.add_edge("chatbot_node", END)

graph = builder.compile()

In [ ]:
message = {"role": "user", "content": "Who walked on the moon for the first time? Print only the name"}
# message = {"role": "user", "content": "What is the latest price of MSFT stock?"}
response = graph.invoke({"messages":[message]})

response["messages"]

In [ ]:
state = None
while True:
    in_message = input("You: ")
    if in_message.lower() in {"quit","exit"}:
        break
    if state is None:
        state: State = {
            "messages": [{"role": "user", "content": in_message}]
        }
    else:
        state["messages"].append({"role": "user", "content": in_message})

    state = graph.invoke(state)
    print("Bot:", state["messages"][-1].content)